In [ ]:
import requests
import pandas as pd
!pip install pyarrow -q
import time
from pathlib import Path

In [ ]:
output_dir = Path("/content/chicago_traffic")
output_dir.mkdir(exist_ok=True)

print(output_dir)

/content/chicago_traffic


In [ ]:


base_url = "https://data.cityofchicago.org/resource/4g9f-3jbs.json"

params = {
    "$select": "segment_id, count(*) as observations",
    "$where": "time >= '2024-03-05T09:20:00' AND time < '2024-03-05T09:30:00'",
    "$group": "segment_id",
    "$order": "segment_id ASC",
    "$limit": 5000
}

r = requests.get(base_url, params=params)

print("STATUS CODE:", r.status_code)
print("\nRESPONSE:")
print(r.text[:2000])

STATUS CODE: 200

RESPONSE:
[{"segment_id":"1","observations":"1"}
,{"segment_id":"2","observations":"1"}
,{"segment_id":"3","observations":"1"}
,{"segment_id":"4","observations":"1"}
,{"segment_id":"5","observations":"1"}
,{"segment_id":"6","observations":"1"}
,{"segment_id":"7","observations":"1"}
,{"segment_id":"8","observations":"1"}
,{"segment_id":"9","observations":"1"}
,{"segment_id":"10","observations":"1"}
,{"segment_id":"11","observations":"1"}
,{"segment_id":"12","observations":"1"}
,{"segment_id":"13","observations":"1"}
,{"segment_id":"14","observations":"1"}
,{"segment_id":"15","observations":"1"}
,{"segment_id":"16","observations":"1"}
,{"segment_id":"17","observations":"1"}
,{"segment_id":"19","observations":"1"}
,{"segment_id":"20","observations":"1"}
,{"segment_id":"21","observations":"1"}
,{"segment_id":"22","observations":"1"}
,{"segment_id":"23","observations":"1"}
,{"segment_id":"24","observations":"1"}
,{"segment_id":"25","observations":"1"}
,{"segment_id":"26","

In [ ]:
test_clean = convert_types(test)

test_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25128 entries, 0 to 25127
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   segment_id                25128 non-null  Int64         
 1   date                      25128 non-null  datetime64[ns]
 2   hour                      25128 non-null  Int64         
 3   street                    25128 non-null  object        
 4   direction                 25128 non-null  object        
 5   from_street               25128 non-null  object        
 6   to_street                 25128 non-null  object        
 7   length                    25128 non-null  float64       
 8   street_heading            25128 non-null  object        
 9   avg_speed                 21503 non-null  float64       
 10  min_speed                 21503 non-null  float64       
 11  max_speed                 21503 non-null  float64       
 12  total_observations

In [ ]:
from datetime import timedelta
import time

start_date = pd.Timestamp("2024-06-11")
end_date = pd.Timestamp("2025-01-01")

current_month = start_date.to_period("M")

while current_month.start_time < end_date:

    month_start = max(
        current_month.start_time,
        start_date
    )

    month_end = min(
        (current_month + 1).start_time,
        end_date
    )

    print(f"\nProcessing {current_month}...")

    daily_frames = []

    current_date = month_start

    while current_date < month_end:

        next_date = current_date + timedelta(days=1)

        start_str = current_date.strftime("%Y-%m-%d")
        end_str = next_date.strftime("%Y-%m-%d")

        try:

            daily_df = get_hourly_traffic(
                start_str,
                end_str
            )

            daily_df = convert_types(daily_df)

            daily_frames.append(daily_df)

            print(
                f"{start_str}: "
                f"{len(daily_df):,} rows"
            )

            # Be polite to the API
            time.sleep(0.3)

        except Exception as e:

            print(
                f"ERROR on {start_str}: {e}"
            )

        current_date = next_date

    # Combine the month's daily data
    month_df = pd.concat(
        daily_frames,
        ignore_index=True
    )

    # Save as Parquet
    file_path = (
        output_dir /
        f"chicago_traffic_{current_month}.parquet"
    )

    month_df.to_parquet(
        file_path,
        index=False
    )

    print(
        f"Saved {file_path.name}: "
        f"{len(month_df):,} rows"
    )

    # Free memory before next month
    del month_df
    del daily_frames

    current_month += 1


Processing 2024-06...
2024-06-11: 11,517 rows
2024-06-12: 25,128 rows
2024-06-13: 25,128 rows
2024-06-14: 25,128 rows
2024-06-15: 23,034 rows
2024-06-16: 25,128 rows
2024-06-17: 25,128 rows
2024-06-18: 25,128 rows
2024-06-19: 25,128 rows
2024-06-20: 25,128 rows
2024-06-21: 25,128 rows
2024-06-22: 25,128 rows
2024-06-23: 25,128 rows
2024-06-24: 25,128 rows
2024-06-25: 25,128 rows
2024-06-26: 25,128 rows
2024-06-27: 25,128 rows
2024-06-28: 25,128 rows
2024-06-29: 25,128 rows
2024-06-30: 25,128 rows
Saved chicago_traffic_2024-06.parquet: 486,855 rows

Processing 2024-07...
2024-07-01: 25,128 rows
2024-07-02: 25,128 rows
2024-07-03: 25,128 rows
2024-07-04: 25,128 rows
2024-07-05: 25,128 rows
2024-07-06: 25,128 rows
2024-07-07: 25,128 rows
2024-07-08: 25,128 rows
2024-07-09: 25,128 rows
2024-07-10: 25,128 rows
2024-07-11: 25,128 rows
2024-07-12: 25,128 rows
2024-07-13: 25,128 rows
2024-07-14: 25,128 rows
2024-07-15: 25,128 rows
2024-07-16: 25,128 rows
2024-07-17: 25,128 rows
2024-07-18: 25

In [ ]:
 parquet_files = sorted(
    output_dir.glob("*.parquet")
)

for file in parquet_files:

    temp = pd.read_parquet(file)

    print(
        file.name,
        f"{len(temp):,} rows",
        temp["date"].min(),
        "→",
        temp["date"].max()
    )

chicago_traffic_2024-06.parquet 486,855 rows 2024-06-11 00:00:00 → 2024-06-30 00:00:00
chicago_traffic_2024-07.parquet 778,968 rows 2024-07-01 00:00:00 → 2024-07-31 00:00:00
chicago_traffic_2024-08.parquet 778,968 rows 2024-08-01 00:00:00 → 2024-08-31 00:00:00
chicago_traffic_2024-09.parquet 748,605 rows 2024-09-01 00:00:00 → 2024-09-30 00:00:00
chicago_traffic_2024-10.parquet 778,968 rows 2024-10-01 00:00:00 → 2024-10-31 00:00:00
chicago_traffic_2024-11.parquet 753,840 rows 2024-11-01 00:00:00 → 2024-11-30 00:00:00
chicago_traffic_2024-12.parquet 776,874 rows 2024-12-01 00:00:00 → 2024-12-31 00:00:00


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import shutil
from pathlib import Path

source_dir = Path("/content/chicago_traffic")

drive_dir = Path(
    "/content/drive/MyDrive/chicago_traffic_project/data"
)

drive_dir.mkdir(
    parents=True,
    exist_ok=True
)

for file in source_dir.glob("*.parquet"):

    shutil.copy2(
        file,
        drive_dir / file.name
    )

print("Files copied to Google Drive:")
for file in sorted(drive_dir.glob("*.parquet")):
    print(file.name)

Files copied to Google Drive:
chicago_traffic_2024-06.parquet
chicago_traffic_2024-07.parquet
chicago_traffic_2024-08.parquet
chicago_traffic_2024-09.parquet
chicago_traffic_2024-10.parquet
chicago_traffic_2024-11.parquet
chicago_traffic_2024-12.parquet


In [ ]:
!pip install duckdb -q
import duckdb
print(duckdb.__version__)

1.3.2


In [ ]:
db_path = (
    "/content/drive/MyDrive/"
    "chicago_traffic_project/"
    "chicago_traffic.duckdb"
)

con = duckdb.connect(db_path)

In [ ]:
parquet_path = (
    "/content/drive/MyDrive/"
    "chicago_traffic_project/data/"
    "*.parquet"
)

In [ ]:

con.execute(f"""
    CREATE OR REPLACE VIEW traffic_raw AS

    SELECT *
    FROM read_parquet('{parquet_path}');
""")

In [ ]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT segment_id) AS segments,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM traffic_raw
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────┬─────────────────────┬─────────────────────┐
│ total_rows │ segments │     first_date      │      last_date      │
│   int64    │  int64   │    timestamp_ns     │    timestamp_ns     │
├────────────┼──────────┼─────────────────────┼─────────────────────┤
│    5103078 │     1047 │ 2024-06-11 00:00:00 │ 2024-12-31 00:00:00 │
└────────────┴──────────┴─────────────────────┴─────────────────────┘



## 1. Data Quality & Profiling

### Question 1: How complete are the hourly speed observations?

In [ ]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        SUM(
            CASE
                WHEN valid_speed_observations = 0 THEN 1
                ELSE 0
            END
        ) AS no_valid_speed_rows,

        ROUND(
            100.0 * SUM(
                CASE
                    WHEN valid_speed_observations = 0 THEN 1
                    ELSE 0
                END
            ) / COUNT(*), 2
        ) AS no_valid_speed_pct

    FROM traffic_raw
""").show()

# results shows there are 16.1% of the results has either null or invalid speed data

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬────────────────────┐
│ total_rows │ no_valid_speed_rows │ no_valid_speed_pct │
│   int64    │       int128        │       double       │
├────────────┼─────────────────────┼────────────────────┤
│    5103078 │              821567 │               16.1 │
└────────────┴─────────────────────┴────────────────────┘



# Question 2: Find out the quality of the data in each hour
Finding out the data quality based on how many valid observations are available within the total observations.

In [ ]:

con.sql("""
    CREATE OR REPLACE VIEW traffic_clean AS

    SELECT
        *,

        valid_speed_observations * 1.0
            / NULLIF(total_observations, 0) AS data_coverage_ratio,

        CASE
            WHEN valid_speed_observations * 1.0
                 / NULLIF(total_observations, 0) >= 2.0/3
                THEN 'High Quality'

            WHEN valid_speed_observations * 1.0
                 / NULLIF(total_observations, 0) >= 1.0/3
                THEN 'Moderate Quality'

            ELSE 'Low Quality'
        END AS data_quality

    FROM traffic_raw
""")

In [ ]:
# find out the percentage of each data quality within the dataset
con.sql("""
    SELECT
        data_quality,
        COUNT(*) AS row_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_rows
    FROM traffic_clean
    GROUP BY data_quality
    ORDER BY row_count DESC
""").show()

# results shows that half of the data provided provides sufficient number of data points within the hour for the calculation to be reasonably accurate

┌──────────────────┬───────────┬─────────────┐
│   data_quality   │ row_count │ pct_of_rows │
│     varchar      │   int64   │   double    │
├──────────────────┼───────────┼─────────────┤
│ High Quality     │   2629116 │       51.52 │
│ Moderate Quality │   1397413 │       27.38 │
│ Low Quality      │   1076549 │        21.1 │
└──────────────────┴───────────┴─────────────┘



# Question 3: When is Chicago traffic worst?
Chicago traffic speeds are lowest during the afternoon peak, particularly from approximately 3–6 PM, with 5 PM showing the lowest citywide average speed.

In [ ]:
con.sql("""
SELECT
    hour,
    AVG(avg_speed) AS hourly_avg_speed
FROM traffic_clean
WHERE data_quality = 'High Quality'
GROUP BY hour
ORDER BY hourly_avg_speed ASC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┬────────────────────┐
│ hour  │  hourly_avg_speed  │
│ int64 │       double       │
├───────┼────────────────────┤
│    17 │ 23.655897230242687 │
│    16 │  23.69263938371656 │
│    15 │  23.93774545963279 │
│    18 │ 24.102060477370078 │
│    14 │  24.37649926955968 │
│     2 │ 24.397850850235635 │
│    13 │  24.48850637870881 │
│    19 │ 24.540695834122072 │
│    12 │   24.6075387550509 │
│    20 │  24.76378562371601 │
│     · │          ·         │
│     · │          ·         │
│     · │          ·         │
│    10 │ 25.078412342870912 │
│    22 │  25.08084674816981 │
│     9 │ 25.163614314936464 │
│    23 │  25.31452991453003 │
│     0 │ 25.353972445785768 │
│     1 │ 25.368518102205165 │
│     6 │ 25.645463598049574 │
│     5 │ 25.814802562630287 │
│     4 │  25.95006382574218 │
│     3 │  26.04476005704241 │
├───────┴────────────────────┤
│     24 rows (20 shown)     │
└────────────────────────────┘



# How do weekday and weekend patterns differ?
---
Weekday traffic shows clearer morning and afternoon slowdowns than weekend traffic. The largest observed weekday-weekend speed gap occurs around the morning commute, with weekend speeds approximately 1.6 mph faster at 8 AM. A second notable gap occurs during the afternoon commute around 4–5 PM.

In [ ]:
con.sql("""
    SELECT
        hour,

        ROUND(AVG(
            CASE
                WHEN DAYOFWEEK(date) NOT IN (0, 6)
                THEN avg_speed
            END
        ),2) AS weekday_avg_speed,

        ROUND(AVG(
            CASE
                WHEN DAYOFWEEK(date) IN (0, 6)
                THEN avg_speed
            END
        ),2) AS weekend_avg_speed,

        ROUND(weekend_avg_speed - weekday_avg_speed,2) AS speed_difference

    FROM traffic_clean

    WHERE data_quality = 'High Quality'
          AND hour BETWEEN 6 AND 22
    GROUP BY hour
    order BY hour


""").show()

┌───────┬───────────────────┬───────────────────┬──────────────────┐
│ hour  │ weekday_avg_speed │ weekend_avg_speed │ speed_difference │
│ int64 │      double       │      double       │      double      │
├───────┼───────────────────┼───────────────────┼──────────────────┤
│     6 │             25.57 │             26.11 │             0.54 │
│     7 │             24.79 │             26.23 │             1.44 │
│     8 │             24.57 │             26.16 │             1.59 │
│     9 │             25.03 │             25.64 │             0.61 │
│    10 │             24.97 │             25.41 │             0.44 │
│    11 │              24.7 │             25.12 │             0.42 │
│    12 │             24.51 │             24.87 │             0.36 │
│    13 │             24.39 │             24.78 │             0.39 │
│    14 │             24.26 │             24.73 │             0.47 │
│    15 │             23.73 │             24.59 │             0.86 │
│    16 │             23.46 │     

### How much slower is each road during rush hour compared with its own normal/off-peak speed?

Segment-specific baseline speeds were calculated using average weekday speeds from 10 AM–1 PM, selected as the relatively stable inter-peak period. Overnight hours were excluded because the analysis focuses on operational conditions during meaningful daytime travel periods.

In [ ]:
# finds out the baseline speed for each segment before calculating the slowdown severity
con.sql("""
WITH baseline AS (


    SELECT
      segment_id,
      hour,
      ROUND(avg(avg_speed),2) AS baseline_speed
    FROM traffic_clean
    WHERE data_quality = 'High Quality'
      AND hour BETWEEN 10 AND 13
    GROUP BY segment_id, hour
)
SELECT
    t.segment_id,
    t.hour,
    t.avg_speed,
    b.baseline_speed,
    ROUND((t.avg_speed - b.baseline_speed)/NULLIF(b.baseline_speed, 0) * 100) AS slowdown_pct
FROM traffic_clean t
JOIN baseline b
    ON t.segment_id = b.segment_id



""").show()

┌────────────┬───────┬────────────────────┬────────────────┬──────────────┐
│ segment_id │ hour  │     avg_speed      │ baseline_speed │ slowdown_pct │
│   int64    │ int64 │       double       │     double     │    double    │
├────────────┼───────┼────────────────────┼────────────────┼──────────────┤
│          1 │    13 │               26.5 │          25.05 │          6.0 │
│          1 │    14 │              28.25 │          25.05 │         13.0 │
│          1 │    15 │               23.5 │          25.05 │         -6.0 │
│          1 │    16 │               26.0 │          25.05 │          4.0 │
│          1 │    17 │               22.4 │          25.05 │        -11.0 │
│          1 │    18 │               25.0 │          25.05 │         -0.0 │
│          1 │    19 │               26.5 │          25.05 │          6.0 │
│          1 │    20 │               26.0 │          25.05 │          4.0 │
│          1 │    21 │               30.0 │          25.05 │         20.0 │
│          1

Relative to segment-specific weekday baseline speeds, Chicago traffic experiences its greatest network-wide slowdown during the afternoon peak. Average slowdown reaches approximately 5.1% at 5 PM, followed closely by 4.9% at 4 PM. Conditions improve substantially after 6 PM.

In [ ]:
con.sql("""
WITH baseline AS (
    SELECT
        segment_id,
        AVG(avg_speed) AS baseline_speed
    FROM traffic_clean
    WHERE data_quality = 'High Quality'
      AND DAYOFWEEK(date) NOT IN (0, 6)
      AND hour BETWEEN 10 AND 13
    GROUP BY segment_id
),
slowdown AS (
    SELECT
        t.segment_id,
        t.hour,
        t.street,
        t.date,
        ROUND(
          (b.baseline_speed - t.avg_speed)
          / NULLIF(b.baseline_speed, 0) * 100
          ,2) AS slowdown_pct
    FROM traffic_clean t
    JOIN baseline b
        ON t.segment_id = b.segment_id
    WHERE t.data_quality = 'High Quality'
      AND DAYOFWEEK(date) NOT IN (0, 6)
      AND t.hour BETWEEN 6 AND 22


)
SELECT
    hour,
    ROUND(AVG(slowdown_pct), 2) AS avg_slowdown_pct

FROM slowdown
GROUP BY hour
ORDER BY avg_slowdown_pct DESC;



""").show()

┌───────┬──────────────────┐
│ hour  │ avg_slowdown_pct │
│ int64 │      double      │
├───────┼──────────────────┤
│    17 │             5.09 │
│    16 │              4.9 │
│    15 │             3.76 │
│    18 │             2.72 │
│    14 │             1.58 │
│    13 │             1.13 │
│    12 │              0.6 │
│    19 │             0.41 │
│     8 │             0.28 │
│    11 │            -0.27 │
│    20 │            -0.33 │
│     7 │            -0.96 │
│    21 │            -1.06 │
│    10 │            -1.44 │
│     9 │            -1.48 │
│    22 │            -2.84 │
│     6 │            -4.41 │
├───────┴──────────────────┤
│ 17 rows        2 columns │
└──────────────────────────┘



During exploratory analysis, three roadway segments had a computed baseline speed of 0 mph. Because slowdown is defined relative to baseline speed, these segments would produce undefined values. They were identified through validation checks and excluded from downstream congestion metrics.
Since these segments had only 1–3 High Quality observations within the baseline time window, they were excluded from subsequent congestion analyses. Segments where observation counts less than 100 are also excluded.

In [ ]:
# created views for critial queries like baseline speed and slowdown frequency
con.sql("""
    CREATE OR REPLACE VIEW baseline AS
    SELECT
        segment_id,
        AVG(avg_speed) AS baseline_speed,
        COUNT(*) AS baseline_obs
    FROM traffic_clean
    WHERE data_quality = 'High Quality'
      AND DAYOFWEEK(date) NOT IN (0, 6)
      AND hour BETWEEN 10 AND 13
      AND avg_speed > 0
    GROUP BY segment_id
    HAVING
      COUNT(*) >= 100
      AND AVG(avg_speed) > 0;



""")

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW slowdown AS
    SELECT
        t.segment_id,
        t.hour,
        t.street,
        t.date,
        ROUND(
            (b.baseline_speed - t.avg_speed)
            / NULLIF(b.baseline_speed, 0) * 100,
            2
        ) AS slowdown_pct,
        t.start_latitude,
        t.start_longitude,
        t.end_latitude,
        t.end_longitude
    FROM traffic_clean t
    JOIN baseline b
        ON t.segment_id = b.segment_id
    WHERE t.data_quality = 'High Quality'
      AND DAYOFWEEK(t.date) NOT IN (0, 6)
      AND t.hour BETWEEN 6 AND 22;



""")

### Which segments experience the largest rush-hour deterioration?
To understand the severity of the slowdown better, I've used various threshold to categorize different level of severity. It provides insights regarding the segments where the slowdown aggrevated further than others.  



In [ ]:
con.sql("""

WITH baseline AS (
    SELECT
    segment_id,
    AVG(avg_speed) AS baseline_speed
  FROM traffic_clean
  WHERE data_quality = 'High Quality' AND
    DAYOFWEEK(date) NOT IN (0, 6) AND
    hour BETWEEN 10 AND 13
  GROUP BY segment_id
  ),
slowdown AS (
    SELECT
        t.segment_id,
        t.hour,
        t.street,
        t.date,
        ROUND(
          (b.baseline_speed - t.avg_speed)
          / NULLIF(b.baseline_speed, 0) * 100
          ,2) AS slowdown_pct,
        t.start_latitude,
        t.start_longitude,
        t.end_latitude,
        t.end_longitude
    FROM traffic_clean t
    JOIN baseline b
        ON t.segment_id = b.segment_id
    WHERE t.data_quality = 'High Quality'
      AND DAYOFWEEK(date) NOT IN (0, 6)
      AND t.hour BETWEEN 6 AND 22


)
SELECT
    segment_id,
    street,
    ROUND(avg(slowdown_pct), 2) AS avg_slowdown_pct,
    ROUND(
      100.0 *COUNT(
        CASE
          WHEN slowdown_pct >= 0 THEN 1
        END
      )/ COUNT(*), 2)
      AS any_slowdown_pct,
    ROUND(
      100.0 *COUNT(
        CASE
          WHEN slowdown_pct >= 10 THEN 1
        END
      )/ COUNT(*), 2)
    AS moderate_slowdown_pct,
    ROUND(
      100.0 *COUNT(
        CASE
          WHEN slowdown_pct >= 20 THEN 1
        END
      )/ COUNT(*), 2)
    AS significant_slowdown_pct,
    ROUND(
      100.0 *COUNT(
        CASE
          WHEN slowdown_pct >= 30 THEN 1
        END
      )/ COUNT(*), 2)
    AS severe_slowdown_pct

FROM slowdown
WHERE DAYOFWEEK(date) NOT IN (0, 6)
GROUP BY street, hour, segment_id;



""")

┌────────────┬─────────┬──────────────────┬──────────────────┬───────────────────────┬──────────────────────────┬─────────────────────┐
│ segment_id │ street  │ avg_slowdown_pct │ any_slowdown_pct │ moderate_slowdown_pct │ significant_slowdown_pct │ severe_slowdown_pct │
│   int64    │ varchar │      double      │      double      │        double         │          double          │       double        │
├────────────┼─────────┼──────────────────┼──────────────────┼───────────────────────┼──────────────────────────┼─────────────────────┤
│          1 │ 55th    │             0.03 │             38.1 │                 14.29 │                     4.76 │                 0.0 │
│          1 │ 55th    │             3.87 │            65.91 │                 18.18 │                      5.3 │                2.27 │
│          1 │ 55th    │             0.72 │            53.49 │                  8.53 │                     1.55 │                 0.0 │
│          1 │ 55th    │            -0.32 │     

Create a view for slowdown is considered severe since resources might not be enough to fix all the problematic corridors, so this view only includes data where slowdown percentage is worse than 20%.

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW segment_summary AS
    SELECT
      segment_id,
      COUNT(*) AS observations,
      street,
      ROUND(AVG(slowdown_pct), 2) AS avg_slowdown_pct,
      ROUND(
          100.0 * COUNT(CASE WHEN slowdown_pct >= 0 THEN 1 END)
          / COUNT(*), 2
      ) AS any_slowdown_pct,
      ROUND(
          100.0 * COUNT(CASE WHEN slowdown_pct >= 10 THEN 1 END)
          / COUNT(*), 2
      ) AS moderate_slowdown_pct,
      ROUND(
          100.0 * COUNT(CASE WHEN slowdown_pct >= 20 THEN 1 END)
          / COUNT(*), 2
      ) AS significant_slowdown_pct,
      ROUND(
          100.0 * COUNT(CASE WHEN slowdown_pct >= 30 THEN 1 END)
          / COUNT(*), 2
      ) AS severe_slowdown_pct,

      start_latitude,
      start_longitude,
      end_latitude,
      end_longitude



    FROM slowdown
    GROUP BY segment_id, street, start_latitude, start_longitude, end_latitude, end_longitude
    ORDER BY significant_slowdown_pct DESC;



""")

Analyzing certain segments' where the weekday speed appears to be suspicious.









In [ ]:
con.sql("""
  SELECT
    segment_id,
    COUNT(*) AS observations,
    MIN(avg_speed) AS min_speed,
    MAX(avg_speed) AS max_speed,
    AVG(avg_speed) AS baseline_speed
  FROM traffic_clean
  WHERE segment_id IN (329,330,1061)
    AND data_quality = 'High Quality'
    AND DAYOFWEEK(date) NOT IN (0,6)
    AND hour BETWEEN 10 AND 13
  GROUP BY segment_id;



""")

┌────────────┬──────────────┬───────────┬───────────┬────────────────┐
│ segment_id │ observations │ min_speed │ max_speed │ baseline_speed │
│   int64    │    int64     │  double   │  double   │     double     │
├────────────┼──────────────┼───────────┼───────────┼────────────────┤
│        330 │            3 │       0.0 │       0.0 │            0.0 │
│       1061 │            2 │       0.0 │       0.0 │            0.0 │
│        329 │            1 │       0.0 │       0.0 │            0.0 │
└────────────┴──────────────┴───────────┴───────────┴────────────────┘

There are certain segemnts where they have less baseline speed available for calculation. With less data available, the results can be less reliable. Hence segemnts with less than 100 observations are omitted.

In [ ]:
con.sql("""
SELECT
    CASE
        WHEN baseline_obs < 10 THEN '<10'
        WHEN baseline_obs < 50 THEN '10-49'
        WHEN baseline_obs < 100 THEN '50-99'
        WHEN baseline_obs < 250 THEN '100-249'
        WHEN baseline_obs < 500 THEN '250-499'
        ELSE '500+'
    END AS bucket,
    COUNT(*) AS segments
FROM (
    SELECT
        segment_id,
        COUNT(*) AS baseline_obs
    FROM traffic_clean
    WHERE data_quality = 'High Quality'
      AND DAYOFWEEK(date) NOT IN (0,6)
      AND hour BETWEEN 10 AND 13
    GROUP BY segment_id
)
GROUP BY bucket
ORDER BY bucket;
""").show()

┌─────────┬──────────┐
│ bucket  │ segments │
│ varchar │  int64   │
├─────────┼──────────┤
│ 10-49   │       45 │
│ 100-249 │      104 │
│ 250-499 │      283 │
│ 50-99   │       30 │
│ 500+    │      525 │
│ <10     │       26 │
└─────────┴──────────┘



### Which corridors should Chicago prioritize? Which corridors are unpredictable?
The Priority Score equally weights normalized average slowdown and the frequency of significant (≥20%) slowdowns. Equal weighting was chosen to balance congestion severity with recurrence. Different agencies may choose different weights depending on operational priorities.


In [ ]:
con.sql("""
  CREATE OR REPLACE VIEW priority_ranking AS
  WITH bounds AS (
    SELECT
      street,
      segment_id,
      avg_slowdown_pct,
      significant_slowdown_pct,
      MIN(avg_slowdown_pct) OVER() AS global_min_slowdown_pct,
      MAX(avg_slowdown_pct) OVER() AS global_max_slowdown_pct,
      MIN(significant_slowdown_pct) OVER () AS global_min_significant_slowdown_pct,
      MAX(significant_slowdown_pct) OVER () AS global_max_significant_slowdown_pct,
      start_latitude,
      start_longitude,
      end_latitude,
      end_longitude


    FROM segment_summary



  ),
  scores as (
    SELECT
      street,
      segment_id,
      avg_slowdown_pct,
      significant_slowdown_pct,
      start_latitude,
      start_longitude,
      end_latitude,
      end_longitude,

      ROUND(
          100* (avg_slowdown_pct-global_min_slowdown_pct)
          / (global_max_slowdown_pct - global_min_slowdown_pct), 2
      ) AS avg_slowdown_score,



      ROUND(
          100.0 * (significant_slowdown_pct-global_min_significant_slowdown_pct)
          / (global_max_significant_slowdown_pct - global_min_significant_slowdown_pct), 2
      ) AS frequency_score

    FROM bounds

  ),

  priority_ranking AS (

  SELECT
      street,
      start_latitude,
      start_longitude,
      end_latitude,
      end_longitude,
      avg_slowdown_score,
      frequency_score,
      segment_id,
      avg_slowdown_pct,
      significant_slowdown_pct,

      ROUND(0.5*(avg_slowdown_score + frequency_score)) AS priority_score
    FROM scores
    ORDER BY priority_score DESC
  )

  SELECT
      segment_id,
      street,
      start_latitude,
      start_longitude,
      end_latitude,
      end_longitude,
      avg_slowdown_pct,
      significant_slowdown_pct,
      avg_slowdown_score,
      frequency_score,
      priority_score,
      ROW_NUMBER() OVER (ORDER BY priority_score DESC) AS priority_rank
  FROM priority_ranking;



""")

In [ ]:
con.sql("""
  SELECT *
FROM priority_ranking
ORDER BY priority_rank;



""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┬────────────────┬─────────────────┬───────────────┬────────────────┬──────────────────┬──────────────────────────┬────────────────────┬─────────────────┬────────────────┬───────────────┐
│ segment_id │    street     │ start_latitude │ start_longitude │ end_latitude  │ end_longitude  │ avg_slowdown_pct │ significant_slowdown_pct │ avg_slowdown_score │ frequency_score │ priority_score │ priority_rank │
│   int64    │    varchar    │     double     │     double      │    double     │     double     │      double      │          double          │       double       │     double      │     double     │     int64     │
├────────────┼───────────────┼────────────────┼─────────────────┼───────────────┼────────────────┼──────────────────┼──────────────────────────┼────────────────────┼─────────────────┼────────────────┼───────────────┤
│       1058 │ Randolph      │  41.8845986991 │   -87.637068542 │ 41.8844304739 │ -87.6474313736 │            18.86 │               

# Key insights

1.   Traffic is the worst around 5pm based on the speed solely via comparison to other hours.
2.   Rush hour (7-8am and 4-5pm) traffic is a lot worse during weekdays than weekends.
3.   Prior to ranking the severity of each corridor, some slowdown percentage were in null. I later discovered that some speed were 0 during the time period where I used to calculate baseline speed.
4.   There are certain segemnts where they have less baseline speed available for calculation. With less data available, the results can be less reliable. Hence segemnts with less than 100 observations are omitted.
5.   The highest priority corridor was ranked by summing the total of the equally weighted normalized average slowdown and frequency of significant slowdowns, with three of the top 10 segments belong to lake shore dr.



In [ ]:

con.sql("SELECT * FROM priority_ranking").to_df().to_csv(
    "priority_ranking.csv", index=False
)

con.sql("SELECT * FROM segment_summary").to_df().to_csv(
    "segment_summary.csv", index=False
)

con.sql("SELECT * FROM slowdown").to_df().to_csv(
    "slowdown.csv", index=False
)